# Board Deployment: Single-Image Gesture Inference

This notebook is the final inference-only version for the AUP board.

It assumes:
- the model is already trained on the laptop
- the board will only do inference
- the input is a single 96x96 image
- required Python libraries will be installed on the board later

Files expected in the same folder:
- `gesture_model.pkl`
- `hand_landmarker.task`
- `test_96x96.jpg`

This notebook does **not** include:
- training
- webcam code
- dataset code
- plotting

In [23]:
from pathlib import Path

PROJECT_DIR = Path(".")

MODEL_PATH = PROJECT_DIR / "gesture_model.pkl"
LANDMARKER_PATH = PROJECT_DIR / "hand_landmarker.task"
IMAGE_PATH = PROJECT_DIR / "thumbs_down_frame_00180.jpg"

COMMAND_MAP = {
    "fist": "FIST",
    "open_palm": "OPEN PALM",
    "two_fingers": "TWO FINGERS",
    "thumbs_up": "THUMBS UP",
    "thumbs_down": "THUMBS DOWN",
}

In [16]:
import sys
import math
import numpy as np
import joblib
import mediapipe as mp
import sklearn

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [17]:
print("Python version:", sys.version)
print("NumPy version:", np.__version__)
print("joblib version:", joblib.__version__)
print("MediaPipe version:", mp.__version__)
print("scikit-learn version:", sklearn.__version__)

Python version: 3.11.5 (main, Sep 11 2023, 08:31:25) [Clang 14.0.6 ]
NumPy version: 2.4.4
joblib version: 1.5.3
MediaPipe version: 0.10.33
scikit-learn version: 1.8.0


In [19]:
missing = []

for p in [MODEL_PATH, LANDMARKER_PATH, IMAGE_PATH]:
    if not p.exists():
        missing.append(str(p))

if missing:
    print("[ERROR] Missing file(s):")
    for m in missing:
        print(" -", m)
else:
    print("[OK] All required files are present.")

[OK] All required files are present.


## Feature extraction

This must match training exactly:

1. translate landmarks so wrist is at origin  
2. scale-normalize by wrist → middle MCP  
3. flatten all normalized `(x, y, z)` values  
4. add geometric fingertip distance features

In [20]:
WRIST = 0
THUMB_TIP = 4
INDEX_TIP = 8
MIDDLE_TIP = 12
RING_TIP = 16
PINKY_TIP = 20
MIDDLE_MCP = 9

def dist3(a, b):
    return math.sqrt(
        (a[0] - b[0]) ** 2 +
        (a[1] - b[1]) ** 2 +
        (a[2] - b[2]) ** 2
    )

def normalize_and_engineer_features(hand_landmarks):
    """
    Returns:
      63 normalized xyz features
      + 9 geometric distance features
    """

    # Step 1: translate so wrist is at origin
    wrist = hand_landmarks[WRIST]
    coords = []
    for lm in hand_landmarks:
        coords.append([
            lm.x - wrist.x,
            lm.y - wrist.y,
            lm.z - wrist.z
        ])

    # Step 2: scale normalize using wrist -> middle MCP
    ref = coords[MIDDLE_MCP]
    scale = math.sqrt(ref[0]**2 + ref[1]**2 + ref[2]**2)
    if scale < 1e-6:
        scale = 1.0

    norm_coords = []
    flat_xyz = []
    for x, y, z in coords:
        nx, ny, nz = x / scale, y / scale, z / scale
        norm_coords.append([nx, ny, nz])
        flat_xyz.extend([nx, ny, nz])

    # Step 3: geometric features
    wrist_pt = norm_coords[WRIST]
    thumb_tip = norm_coords[THUMB_TIP]
    index_tip = norm_coords[INDEX_TIP]
    middle_tip = norm_coords[MIDDLE_TIP]
    ring_tip = norm_coords[RING_TIP]
    pinky_tip = norm_coords[PINKY_TIP]

    geo = [
        dist3(wrist_pt, thumb_tip),   # wrist_to_thumb_tip
        dist3(wrist_pt, index_tip),   # wrist_to_index_tip
        dist3(wrist_pt, middle_tip),  # wrist_to_middle_tip
        dist3(wrist_pt, ring_tip),    # wrist_to_ring_tip
        dist3(wrist_pt, pinky_tip),   # wrist_to_pinky_tip
        dist3(thumb_tip, index_tip),  # thumb_index_tip_dist
        dist3(index_tip, middle_tip), # index_middle_tip_dist
        dist3(middle_tip, ring_tip),  # middle_ring_tip_dist
        dist3(ring_tip, pinky_tip),   # ring_pinky_tip_dist
    ]

    return np.array(flat_xyz + geo, dtype=np.float32).reshape(1, -1)

In [21]:
model = joblib.load(MODEL_PATH)
print("[OK] Loaded Random Forest model.")

base_options = python.BaseOptions(model_asset_path=str(LANDMARKER_PATH))
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

landmarker = vision.HandLandmarker.create_from_options(options)
print("[OK] Loaded MediaPipe Hand Landmarker.")

[OK] Loaded Random Forest model.
[OK] Loaded MediaPipe Hand Landmarker.


I0000 00:00:1776699031.934089 35223432 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1776699031.938353 35223434 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776699031.942811 35223434 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [24]:
mp_image = mp.Image.create_from_file(str(IMAGE_PATH))
result = landmarker.detect(mp_image)

if not result.hand_landmarks:
    print("No hand detected in image.")
else:
    hand_landmarks = result.hand_landmarks[0]
    features = normalize_and_engineer_features(hand_landmarks)

    pred_label = model.predict(features)[0]

    pred_conf = None
    if hasattr(model, "predict_proba"):
        pred_conf = float(np.max(model.predict_proba(features)))

    command = COMMAND_MAP.get(pred_label, "UNKNOWN")

    print("Prediction:", pred_label)
    print("Command:", command)
    if pred_conf is not None:
        print("Confidence:", round(pred_conf, 4))

Prediction: thumbs_down
Command: THUMBS DOWN
Confidence: 1.0


/Users/ananyavirmani/Documents/UW Madison/Spring 2026/ECE 554/capstone/gesture_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/ananyavirmani/Documents/UW Madison/Spring 2026/ECE 554/capstone/gesture_env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
